# 调试 ROS2 数据包裁剪后生成空 .db3 文件的问题

本 Notebook 演示如何重现裁剪 0‑0.9s 后导出得到空 `.db3` 的情况，分析根本原因，并提供修复示例。

---

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import tempfile

# rosbags 可能在当前 notebook 环境中不存在，需要先安装
try:
    from rosbags.highlevel import AnyReader
    from rosbags.rosbag2 import Writer as Db3Writer
except ImportError:
    print("rosbags not available in this kernel")

## 2. 加载并检查导出生成的 .db3/.yaml 文件

假设你已经从工具导出了裁剪包并解压到某个临时目录，下面我们读取 `metadata.yaml` 查看其内容。

In [ ]:
import yaml

# 指定假定的解压目录路径
output_dir = Path("/tmp/exported_db3")  # 修改为实际路径
yaml_path = output_dir / "metadata.yaml"
if yaml_path.exists():
    print(yaml_path.read_text())
else:
    print("metadata.yaml not found")


## 3. 使用 rosbags 读取 db3 文件并查看 topics

如果包确实未写入消息，`AnyReader` 打开后 `reader.topics` 为空，`reader.message_count` 为 0。

In [ ]:
db3_path = output_dir / "output_db3.db3"
if db3_path.exists():
    with AnyReader([db3_path]) as r:
        print("topics:", r.topics)
        print("message_count", r.message_count)
        print("duration", r.duration)
else:
    print("db3 file not found")

## 4. 分析导出逻辑中可能导致空 .db3 的问题

工具代码中根据 `reader.start_time` 来偏移过滤边界，如果该字段为 0，
`filter_end_ns = start_time_ns + int(range_sec[1] * 1e9)` 就会落在很小的值，
而实际消息时间戳通常远大于 0，结果没有任何消息满足过滤条件。

In [ ]:
# 简单重现过滤逻辑

def check_filter(start_time_ns, duration_ns, range0, range1):
    if range0 <= 0.01:
        filter_start_ns = 0
    else:
        filter_start_ns = start_time_ns + int(range0 * 1e9)
    if range1 >= duration_ns - 0.01:
        filter_end_ns = 2**63 - 1
    else:
        filter_end_ns = start_time_ns + int(range1 * 1e9)
    return filter_start_ns, filter_end_ns

# 设定 start_time_ns 为 0，duration 100s
print(check_filter(0, 100*1e9, 0, 0.9))
# 结果 filter_end_ns 为 900000000 ，远小于真实消息时间戳 (>1e18)

## 5. 修改裁剪时间边界处理并重新导出

我们可以在读取包时检测 `start_time_ns` 是否为 0。如果是，就遍历第一条消息获取真实起始时间。
下面是一个修复后的示例 `crop` 函数。

In [ ]:
def crop(bag_path, start_s, end_s, out_dir):
    # 读取包获取起始时间
    with AnyReader([bag_path]) as r:
        start_time = r.start_time or 0
        if start_time == 0:
            # 读取第一条消息作为基准
            for _, ts, _ in r.messages():
                start_time = ts
                break
        duration = r.duration or 0

    # 计算过滤边界
    if start_s <= 0.01:
        filter_start = start_time
    else:
        filter_start = start_time + int(start_s * 1e9)
    if end_s >= (duration * 1e-9) - 0.01:
        filter_end = 2**63 - 1
    else:
        filter_end = start_time + int(end_s * 1e9)

    with Db3Writer(out_dir, version=9) as w:
        conn_map = {}
        with AnyReader([bag_path]) as r2:
            for c in r2.connections:
                conn_map[c.id] = w.add_connection(
                    topic=c.topic, msgtype=c.msgtype,
                    typestore=r2.typestore, digest=c.digest
                )
            for conn, ts, raw in r2.messages():
                if filter_start <= ts <= filter_end:
                    w.write(conn_map[conn.id], ts, raw)


## 6. 验证修复后的导出

调用上面的 `crop` 函数，解压结果，再次用 AnyReader 打开检查是否有 topics。

In [ ]:
# 假定存在原始 bag 和临时输出目录
# original_bag = Path("/tmp/original.bag")
# crop(original_bag, 0, 0.9, Path("/tmp/fixed_out"))
# with AnyReader([Path("/tmp/fixed_out")]) as r:
#     print(r.topics, r.message_count)


## 7. 示例：生成并上传压缩包进行解析

下面的伪代码展示如何把裁剪输出打包成 zip 并让原工具解析（模拟上传）。

In [ ]:
# 这段代码仅作说明，真实上传过程由 Streamlit UI 触发

dir_to_zip = Path("/tmp/fixed_out")
zip_path = shutil.make_archive(str(dir_to_zip), 'zip', dir_to_zip)
print(f"生成压缩包: {zip_path}")

# 在工具界面选择“上传文件”，并上传此 zip。
# 工具内部会把它解压到临时目录然后调用 AnyReader 读取。
